# Support Ticket Classifier

**Ticket-type routing with human handoff · Python · Scikit-learn · Logistic Regression**

## Hero Overview

A text classifier that routes support tickets by type (Incident, Request, Problem, Change) from the customer's subject and body alone, built on 16,338 English tickets drawn from a 28,587-ticket synthetic multilingual set.

Family-grouped 4-fold validation keeps reworded near-duplicates on one side of every split, and a confidence threshold auto-routes the clear cases while sending the rest to people.

In [1]:
outcomes = {
    "Test Accuracy": "79.9% (95% CI 75.2\u201384.5)",
    "Macro F1": "0.815",
    "Auto-Routed": "46% at 96.4% accuracy",
    "Request / Change Recall": "98.5% / 96.7%",
    "Incident / Problem Recall": "72.9% / 59.7%",
    "Near-Duplicate Families": "7,985 from 16,338 tickets",
    "Department Target": "38.4% grouped-CV, dropped"
}

pd.DataFrame(outcomes.items(), columns=["Metric", "Value"])

Metric,Value
Test Accuracy,79.9% (95% CI 75.2–84.5)
Macro F1,0.815
Auto-Routed,46% at 96.4% accuracy
Request / Change Recall,98.5% / 96.7%
Incident / Problem Recall,72.9% / 59.7%
Near-Duplicate Families,"7,985 from 16,338 tickets"
Department Target,"38.4% grouped-CV, dropped"


In [2]:
# Radar (skill intensity) + Bubble cluster (implementation strength).
fig

Implementation Strength by System

In [3]:
# Technology Capability Network — concepts ↔ tools (bipartite, D3).

Support Ticket Classifier Network 
 Text feature engineering, family-grouped validation, and confidence-based routing for ticket-type classification.

# Storyline

## Context

Support tickets arrive as free text and need routing. The first target was department, but department labels proved unreliable: near-identical tickets landed in different departments, and an honest family-grouped evaluation scored 38.4%, only 9 points above the 29.3% floor of always guessing the largest class. The target moved to ITIL ticket type (Incident, Request, Problem, Change), which the text alone can predict. The data is a synthetic multilingual Kaggle set, so results need real-world validation before deployment. See the [write-up](https://writing.karthikiyer.info/the-model-was-fine-the-labels-werent/) and the [live notebook](https://projects.karthikiyer.info/#tickets).

## Workflow

Language filter (16,338 English of 28,587) → text cleaning and placeholder masking → subject + body combined → word and word-pair TF-IDF → near-duplicate families → family-grouped 4-fold validation on train → model comparison → one test-set read → confidence threshold for routing.

## Technical Implementation

- **Cleaning:** newlines and `<br>` replaced with spaces; emails, URLs, phone and account numbers masked to placeholders (`<email>`, `<url>`, `<tel_num>`, `<acc_num>`, `<num>`). No stemming or stop-word removal.
- **Features:** subject and body combined, then word and word-pair TF-IDF (~43K features).
- **Validation:** tickets with text similarity ≥0.60 grouped into 7,985 families via SciPy connected components; family-grouped 4-fold on train so no family crosses a split. The test set is opened once, at the end.
- **Model:** four model families compared under pre-registered rules (accuracy decides, ties within one point break on macro F1); Logistic Regression with balanced class weights won, and a 30-combination sweep moved accuracy by only −0.2 points.
- **Routing:** predictions at or above 0.75 confidence are auto-routed; the rest escalate to a person.

## Outcomes

Test accuracy was 79.9% (95% CI 75.2–84.5) with 0.815 macro F1; 46% of tickets were auto-routed at 96.4% accuracy. Request (98.5%) and Change (96.7%) recall were strong, Incident (72.9%) moderate, and Problem (59.7%) weak. The model is English-only, does not route to departments, and is unreliable on tickets under ten words.